# Data Acquisition

This notebook acquires the **raw data** for the *Grocery Product Placement Analysis* project (Group 11, ZHAW PODSV FS2026). It downloads the Instacart Online Grocery Shopping dataset (2017) from Kaggle, places the CSV files in the project's `data/` folder, and validates that the expected files were acquired correctly.

Running this notebook is the first step for any new team member: it makes the raw data available locally so that the exploratory analysis, the documentation, and the dashboard can be reproduced.

> **Scope.** This notebook handles *data acquisition only*. The processed data layer (product ranking and product co-purchase pairs) is built separately in `evaluation/graph_plot.ipynb`, using the helper functions in `eda/defs_graph_plot.py`.

## Setup

Locate the project root (so the notebook works regardless of the directory it is launched from) and make sure the `data/` folder exists.

In [2]:
import shutil
from pathlib import Path

import kagglehub
import pandas as pd


# Walk up from the current working directory until pyproject.toml is found.
# This avoids hard-coded relative paths such as "../data", which break when
# the notebook is run from a different directory.
def find_project_root(marker="pyproject.toml"):
    for directory in [Path.cwd(), *Path.cwd().parents]:
        if (directory / marker).exists():
            return directory
    raise FileNotFoundError(
        f"Could not locate the project root (no '{marker}' found above {Path.cwd()})."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data folder:  {DATA_DIR}")

Project root: /Users/larapestrin/Documents/ZHAW/PODSV_Project
Data folder:  /Users/larapestrin/Documents/ZHAW/PODSV_Project/data


## Download the raw data

Download the dataset into the local `kagglehub` cache and copy every file into the project's `data/` folder.

In [3]:
KAGGLE_DATASET = "yasserh/instacart-online-grocery-basket-analysis-dataset"

# Download to the local kagglehub cache and return the cache path.
download_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))

# Copy every downloaded file into the project's data/ folder.
copied = []
for file in sorted(download_path.iterdir()):
    if file.is_file():
        shutil.copy(file, DATA_DIR / file.name)
        copied.append(file.name)

print(f"Copied {len(copied)} file(s) to {DATA_DIR}:")
for name in copied:
    print(f"  - {name}")

Copied 6 file(s) to /Users/larapestrin/Documents/ZHAW/PODSV_Project/data:
  - aisles.csv
  - departments.csv
  - order_products__prior.csv
  - order_products__train.csv
  - orders.csv
  - products.csv


## Validate the acquired data

Confirm that the files this project depends on are present and readable. `order_products__train.csv` is part of the Kaggle dataset but is the competition evaluation set and is not used in this project.

In [4]:
EXPECTED_FILES = [
    "orders.csv",
    "order_products__prior.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv",
]

print("Validation of acquired data")
print("-" * 60)

all_ok = True
for name in EXPECTED_FILES:
    path = DATA_DIR / name
    if not path.exists():
        print(f"  MISSING  {name}")
        all_ok = False
        continue
    size_mb = path.stat().st_size / 1024 ** 2
    # Read only the first rows to confirm the file is a readable CSV.
    head = pd.read_csv(path, nrows=5)
    print(f"  OK       {name:<28} {size_mb:8.1f} MB   columns: {list(head.columns)}")

print("-" * 60)
if all_ok:
    print("All expected files are present and readable.")
else:
    raise FileNotFoundError("Some expected files are missing - see the list above.")

Validation of acquired data
------------------------------------------------------------
  OK       orders.csv                      103.9 MB   columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
  OK       order_products__prior.csv       550.8 MB   columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']
  OK       products.csv                      2.1 MB   columns: ['product_id', 'product_name', 'aisle_id', 'department_id']
  OK       aisles.csv                        0.0 MB   columns: ['aisle_id', 'aisle']
  OK       departments.csv                   0.0 MB   columns: ['department_id', 'department']
------------------------------------------------------------
All expected files are present and readable.


## Next steps

The raw data is now available in the project's `data/` folder. This folder is excluded from version control (see `.gitignore`), so every team member runs this notebook once to obtain the data locally.

- **Exploratory data analysis:** `eda/01_eda.ipynb`
- **Processed data layer** (product ranking and co-purchase pairs): `evaluation/graph_plot.ipynb`
- **Documentation:** `docs/data_report.qmd`